# 02: Your own object (task 5)

Same pipeline, your data. The labels now come from a **text prompt**: Grounding DINO finds boxes
for your words, SAM turns them into masks. Nobody checks them but you.

Every experiment: ✍️ prediction, run, ✍️ explanation.

In [ ]:
# Setup: installs what is missing (Colab / RunPod) and finds vision.py
import importlib.util, pathlib, subprocess, sys, time, urllib.request
for pkg, mod in [("transformers", "transformers"), ("ultralytics", "ultralytics"),
                 ("segmentation-models-pytorch", "segmentation_models_pytorch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if (pathlib.Path.cwd().parent / "vision.py").exists():   # opened from vision/notebooks or vision/sol
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
elif not pathlib.Path("vision.py").exists():            # Colab: fetch it from the course repo
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhaw-physical-ai/cas-pair/main/vision/vision.py", "vision.py")
from vision import *
device = get_device()
print("device:", device)
if device == "cuda":   # YOLO needs torchvision's NMS on the GPU; some new GPUs (Blackwell) lack its code
    import torchvision
    try:
        torchvision.ops.nms(torch.tensor([[0., 0, 1, 1]], device="cuda"), torch.ones(1, device="cuda"), 0.5)
    except Exception as e:
        print("WARNING: YOLO will crash on this GPU:", e, "\nOn RunPod: deploy an RTX A4000, L4 or RTX 4090 instead (RUNPOD.md)")

## Your photos
The notebook is set up for a **laptop**: everyone has one with them. Put 30-50 photos of it in
`my_photos/`, and 10 photos taken in a *different place* in `my_photos_test/` (Colab: upload a
zip in the file panel, then `!unzip -q photos.zip -d my_photos`). Vary the background, angle,
distance, light, and open / closed. Another object works too: change `PROMPT` and `NAME`.

In [ ]:
PROMPT = "a laptop."     # your object: lower case, one noun phrase, ends with a period
NAME = "laptop"

photos = load_folder("my_photos")
test_photos = load_folder("my_photos_test")
print(len(photos), "training photos,", len(test_photos), "test photos")
fig, axs = plt.subplots(1, 6, figsize=(24, 4))
for ax, im in zip(axs, photos):
    show(ax, im)

## From words to boxes: Grounding DINO
In tasks 1-4 a human drew the boxes, and SAM turned them into masks. Here nobody has drawn
anything, and SAM 2.1 cannot help: it takes clicks and boxes, not words. So a second foundation
model goes in front of it.

**Grounding DINO** is an *open-vocabulary detector*. A normal detector like YOLO knows a fixed
list of classes (COCO's 80) and nothing else. Grounding DINO takes the image **and a text**, and
returns a box for every region that matches the words, with a score per box. It was trained on
millions of images with captions and phrases linked to boxes, so it has learned to connect words
to image regions: you can ask for "a laptop.", "a red cup." or "a person wearing a helmet."
without training anything.

The pipeline for your labels:
```
photo + "a laptop."  →  Grounding DINO  →  boxes + scores  →  SAM 2.1  →  one mask per box
```
`TextBoxer` from `vision.py` wraps it: `boxes, scores = boxer(image, PROMPT, threshold=0.35)`.
- the prompt: lower case, one noun phrase, ends with a period;
- `threshold`: the minimum score for a box. Lower finds more, including wrong things; higher
  misses more. Try 0.2-0.5;
- it is slow (~0.1-0.5 s per image) and large (~700 MB): fine for labeling, not for the robot's
  loop. That is why we still train a small YOLO afterwards.

It also makes mistakes: a similar object, only a part of the object, or nothing at all. That is
why you check every label below.

In [ ]:
boxer = TextBoxer()     # Grounding DINO tiny, ~700 MB on first use
sam = SamLabeler()
boxes, scores = boxer(photos[0], PROMPT)
fig, ax = plt.subplots(figsize=(6, 6))
show(ax, photos[0], sam(photos[0], boxes=boxes)[0] if boxes else None, boxes=boxes, title=f"scores {np.round(scores, 2)}")

## Label
For every photo: boxes from the prompt, a mask from SAM.

**✍️ Your prediction (before running anything):** How many of your photos will get a wrong or missing label?

> 

In [ ]:
# your code

## Check every label
This is the human part of the pipeline: look at every label and drop the bad ones.

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** Which labels did you drop, and why?

> 

## Train and test
Train YOLO on the kept photos (see task 3), then try it on `test_photos`: taken in a place the
model has never seen.

**✍️ Your prediction (before running anything):** Will it find your object in the new place? What will it confuse it with?

> 

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** Where does it fail, and why?

> 

## Advanced
- How many photos did you really need? Train on half of `keep` and compare on the test photos.
- The test labels also came from SAM. How sure are you of your test numbers?
- A laptop is one of COCO's 80 classes, so YOLO knew laptops before you trained it. Run the
  untrained `YOLO("yolo11n-seg.pt")` on your test photos: what did your training add? For an
  object COCO does not know (a specific tool or part), it has to learn everything from your photos.